In [1]:
# import necessary libraries
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Energy balances Eurostat

In [2]:
# Load the CSV
df_energy_balance = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/Energy balances - sankey ready.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_energy_balance["Source"], df_energy_balance["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_energy_balance["source_id"] = df_energy_balance["Source"].map(node_dict)
df_energy_balance["target_id"] = df_energy_balance["Target"].map(node_dict)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_energy_balance["source_id"],
        target=df_energy_balance["target_id"],
        value=df_energy_balance["Value tonne Carbon"]
    )

)])

fig.update_layout(title_text="Carbon Flow Energy Balances EU")

fig.show()

## Biomass 2022 EU

In [3]:
# Load the CSV
df_biomass_2022 = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/biomass_uses_and_flows_2022.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_biomass_2022["Source"], df_biomass_2022["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_biomass_2022["source_id"] = df_biomass_2022["Source"].map(node_dict)
df_biomass_2022["target_id"] = df_biomass_2022["Target"].map(node_dict)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_biomass_2022["source_id"],
        target=df_biomass_2022["target_id"],
        value=df_biomass_2022["Value C (TONNES)"]
    )

)])

fig.update_layout(title_text="Carbon Flow Biomass Balances EU")

fig.show()




In [4]:
# Load the CSV
df_biomass_2022 = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/biomass_uses_and_flows_2022_2.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_biomass_2022["Source"], df_biomass_2022["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_biomass_2022["source_id"] = df_biomass_2022["Source"].map(node_dict)
df_biomass_2022["target_id"] = df_biomass_2022["Target"].map(node_dict)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_biomass_2022["source_id"],
        target=df_biomass_2022["target_id"],
        value=df_biomass_2022["Value C (TONNES)"]
    )

)])

fig.update_layout(title_text="Carbon Flow Biomass Balances EU")

fig.show()


## Fossil Carbon (own calculations), aggregated product flow

In [5]:
# Load the CSV
df_fossil = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/Sankey diagram fossil.csv")

# create unique node list
all_nodes = pd.concat([df_fossil["Source"], df_fossil["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_fossil["source_id"] = df_fossil["Source"].map(node_dict)
df_fossil["target_id"] = df_fossil["Target"].map(node_dict)

carrier_colors = {
    "Crude oil": "rgba(0, 0, 0, 0.6)",       # black
    "Oil": "rgba(0, 0, 0, 0.6)",
    "Gas": "rgba(255, 165, 0, 0.6)",         # orange
    "Hard coal": "rgba(178, 65, 0, 0.8)",    # ugly brown
    "Brown coal": "rgba(139, 69, 19, 0.8)",  # brown
    "Peat": "rgba(160, 82, 45, 0.8)",        # lighter brown
    "Mixed": "rgba(11, 60, 187, 0.4)"      # blue for aggregated flows
}
#calculate outflows per source and target nodes
source_totals = df_fossil.groupby("Source")["Value_tC"].sum()
node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_fossil["color"] = df_fossil["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_fossil["source_id"],
        target=df_fossil["target_id"],
        value=df_fossil["Value_tC"],
        color=df_fossil["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Fossil Carbon Flows (2023)",
    font_size=12
)
fig.update_layout(
    width=1400,
    height=900
)
fig.update_layout(
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    )
)
fig.update_layout(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)


fig.show()

In [6]:
# balance check
# some flows should not match, inflows/outflows/sinks have only one in or outflow. carbon accumulates there or leaves the system. 

node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()
node_outflow = df_fossil.groupby("Source")["Value_tC"].sum()

balance_df_fossil = pd.DataFrame({
    "inflow": node_inflow,
    "outflow": node_outflow
}).fillna(0)

balance_df_fossil["difference"] = balance_df_fossil["inflow"] - balance_df_fossil["outflow"]

imbalanced_nodes = balance_df_fossil[abs(balance_df_fossil["difference"]) > 1e-3]
print(imbalanced_nodes)

                             inflow       outflow    difference
CO2 captured           1.400136e+04  0.000000e+00  1.400136e+04
Carbon Emissions       6.209556e+08  0.000000e+00  6.209556e+08
Exports                5.477100e+07  0.000000e+00  5.477100e+07
Fossil inflow          8.407547e+08  8.407547e+08  2.851480e+01
Import                 0.000000e+00  6.788781e+08 -6.788781e+08
Indigenous production  0.000000e+00  1.561214e+08 -1.561214e+08
International bunkers  6.587335e+07  0.000000e+00  6.587335e+07
Products?              9.527839e+07  0.000000e+00  9.527839e+07
Released stocks        0.000000e+00  5.755170e+06 -5.755170e+06
Stock                  3.862326e+06  0.000000e+00  3.862326e+06
Transformation         7.818430e+08  7.818430e+08  9.999990e-03


In [7]:
# system wide check

total_inflows = df_fossil[df_fossil["flow_type"] == "Inflow"]["Value_tC"].sum()
total_outflows = df_fossil[df_fossil["flow_type"] == "Outflow"]["Value_tC"].sum()
total_emissions = df_fossil[df_fossil["flow_type"] == "Emissions"]["Value_tC"].sum()
total_stock = df_fossil[df_fossil["flow_type"] == "Stock"]["Value_tC"].sum()
total_sink = df_fossil[df_fossil["flow_type"] == "Sink"]["Value_tC"].sum()

print(total_inflows)
print(total_outflows + total_emissions + total_stock + total_sink)

840754680.8059999
840754652.2812002


# Final: Fossil flows (including detailed product outflow)

In [8]:
df_fossil = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/sankey_files/Fossil_Sankey_Final.csv")

# create unique node list
all_nodes = pd.concat([df_fossil["Source"], df_fossil["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_fossil["source_id"] = df_fossil["Source"].map(node_dict)
df_fossil["target_id"] = df_fossil["Target"].map(node_dict)

carrier_colors = {
    "Oil":                "rgba(20, 20, 20, 0.9)",        # darker → strong black in greyscale
    "Gas":                "rgba(255, 165, 0, 0.85)",      # keep orange, slightly अधिक opaque
    "Coal":               "rgba(70, 50, 35, 0.9)",        # darker brown → separates from refinery
    "Refinery Products":  "rgba(170, 120, 70, 0.8)",      # slightly lighter than before
    "Fossil materials":   "rgba(210, 180, 120, 0.6)",     # lighter + more transparent
    "Carbon Emissions":   "rgba(200, 30, 30, 0.9)"        # darker red → strong contrast in grey
}

df_fossil["color"] = df_fossil["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

#calculate outflows per source and target nodes
source_totals = df_fossil.groupby("Source")["Value_tC"].sum()
node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_fossil["source_id"],
        target=df_fossil["target_id"],
        value=df_fossil["Value_tC"],
        color=df_fossil["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Fossil Carbon Flows (2023)",
    font_size=17,
    width=1700,
    height=1000,
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    ),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()

## Sensitivity Analysis: Fossil Carbon (values from nrg_bal sheet)

In [20]:
# Load the CSV
df_fossil = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/Sankey - Fossil_nrg_bal.csv")

# create unique node list
all_nodes = pd.concat([df_fossil["Source"], df_fossil["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_fossil["source_id"] = df_fossil["Source"].map(node_dict)
df_fossil["target_id"] = df_fossil["Target"].map(node_dict)

carrier_colors = {
    "Oil and petroleum products (excluding biofuel portion)": "rgba(20, 20, 20, 0.9)",   # true dark anchor

    "Oil shale and oil sands": "rgba(120, 20, 30, 0.9)",   # dark red-brown (distinct from oil)

    "Natural gas": "rgba(255, 170, 40, 0.85)",   # brighter → mid-light grey

    "Peat and peat products": "rgba(110, 70, 30, 0.9)",   # earthy brown (mid-dark, but lighter than oil)

    "Mixed": "rgba(140, 160, 200, 0.6)",   # light desaturated blue → light grey

    "Solid fossil fuels": "rgba(90, 40, 110, 0.9)"   # dark purple → dark-mid tone
}

#calculate outflows per source and target nodes
source_totals = df_fossil.groupby("Source")["Value_tC"].sum()
node_inflow = df_fossil.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_fossil["color"] = df_fossil["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_fossil["source_id"],
        target=df_fossil["target_id"],
        value=df_fossil["Value_tC"],
        color=df_fossil["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Fossil Carbon Flows (2023)",
    font_size=12
)
fig.update_layout(
    width=1400,
    height=900
)
fig.update_layout(
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    )
)
fig.update_layout(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)


fig.show()

## Biomass

In [10]:
# Load the CSV
df_biomass = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/sankey_files/Biogenic_Sankey_Final.csv")

# create unique node list
all_nodes = pd.concat([df_biomass["Source"], df_biomass["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_biomass["source_id"] = df_biomass["Source"].map(node_dict)
df_biomass["target_id"] = df_biomass["Target"].map(node_dict)

carrier_colors = {
    "Crops": "rgba(140, 210, 90, 0.85)",     # lighter → clearly light grey
    "Wood": "rgba(30, 80, 30, 0.95)",        # darker → strong dark anchor
    "Animal Products": "rgba(200, 160, 80, 0.85)",  # slightly darker ochre
    "Products from biomass": "rgba(40, 140, 160, 0.85)",  # slightly darker teal
    "Mixed Biogenic": "rgba(130, 130, 130, 0.65)",  # pushed toward neutral grey
    "Carbon Emissions (Biogenic)": "rgba(210, 90, 30, 0.9)",  # darker orange → stronger contrast
}

#calculate outflows per source and target nodes
source_totals = df_biomass.groupby("Source")["Value_tC"].sum()
node_inflow = df_biomass.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_biomass["color"] = df_biomass["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_biomass["source_id"],
        target=df_biomass["target_id"],
        value=df_biomass["Value_tC"],
        color=df_biomass["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Biogenic Carbon Flows (2023)",
    font_size=17,
    width=1700,
    height=1000,
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    ),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)


fig.show()

# Biomass from Eurostat balances (unbalanced)

In [11]:
# Load the CSV
df_biomass_2022 = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/biomass_uses_and_flows_2022_2.csv")

# Create list of unique nodes
nodes = list(pd.concat([df_biomass_2022["Source"], df_biomass_2022["Target"]]).unique())

# Create dictionary mapping node name -> index
node_dict = {name: i for i, name in enumerate(nodes)}

# Map names to numeric IDs
df_biomass_2022["source_id"] = df_biomass_2022["Source"].map(node_dict)
df_biomass_2022["target_id"] = df_biomass_2022["Target"].map(node_dict)

#calculate outflows per source and target nodes
source_totals = df_biomass_2022.groupby("Source")["Value C (TONNES)"].sum()
node_inflow = df_biomass_2022.groupby("Target")["Value C (TONNES)"].sum()

labels_with_values = []
for node in nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

# Create Sankey
fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        label=nodes,
        pad=20,
        thickness=20
    ),
    
    link=dict(
        source=df_biomass_2022["source_id"],
        target=df_biomass_2022["target_id"],
        value=df_biomass_2022["Value C (TONNES)"]
    )

)])

fig.update_layout(title_text="Carbon Flow Biomass Balances EU")

fig.show()


--> nu wil ik de values bij deze uiteindes en dan moet ik de twee diagrammen met elkaar vergelijken. 
Dan kan ik kijken of er verschillen zijn in de flows en of er misschien nog iets ontbreekt in de ene of de andere dataset.

# Limestone

In [12]:
# Load the CSV
df_limestone = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/sankey limestone.csv")

# create unique node list
all_nodes = pd.concat([df_limestone["Source"], df_limestone["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_limestone["source_id"] = df_limestone["Source"].map(node_dict)
df_limestone["target_id"] = df_limestone["Target"].map(node_dict)

carrier_colors = {
    "Limestone": "rgba(150, 180, 19, 0.7)",       # yellow
    "Chalk": "rgba(10, 10, 190, 0.8)",       # dark blue
    "Mixed": "rgba(128, 128, 128, 0.5)",  # blue
    "CO2 ": "rgba(150, 0, 100, 1)"  # purple
}

#calculate outflows per source and target nodes
source_totals = df_limestone.groupby("Source")["Value_tC"].sum()
node_inflow = df_limestone.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_limestone["color"] = df_limestone["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_limestone["source_id"],
        target=df_limestone["target_id"],
        value=df_limestone["Value_tC"],
        color=df_limestone["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Limestone Carbon Flows (2023)",
    font_size=12
)
fig.update_layout(
    width=1400,
    height=900
)
fig.update_layout(
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    )
)
fig.update_layout(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)


fig.show()

# Final Mineral Flows 

In [13]:
# Load the CSV
df_limestone = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/sankey_files/Limestone_Sankey_Final.csv")

# create unique node list
all_nodes = pd.concat([df_limestone["Source"], df_limestone["Target"]]).unique()
node_dict = {node: i for i, node in enumerate(all_nodes)}

# map to numeric IDs
df_limestone["source_id"] = df_limestone["Source"].map(node_dict)
df_limestone["target_id"] = df_limestone["Target"].map(node_dict)

carrier_colors = {
    "Limestone (Manufacturing)": "rgba(120, 140, 190, 0.85)",  # clearer blue (industrial)
    "Limestone (Aggregates)":    "rgba(200, 185, 140, 0.80)",  # sandy beige (construction feel)
    "Chalk":                     "rgba(140, 150, 170, 0.80)",  # near-white (but brighter)
    "Dolomite":                  "rgba(140, 120, 170, 0.85)",  # subtle purple-grey (distinct)
    "Mixed Mineral":             "rgba(150, 140, 130, 0.75)",  # neutral warm grey (mix)
    "Carbon Emissions":          "rgba(220, 50, 50, 0.75)",    # keep consistent with fossil
}

#calculate outflows per source and target nodes
source_totals = df_limestone.groupby("Source")["Value_tC"].sum()
node_inflow = df_limestone.groupby("Target")["Value_tC"].sum()

labels_with_values = []
for node in all_nodes:
    inflow = node_inflow.get(node, 0)
    outflow = source_totals.get(node, 0)
    # choose what to display
    value = inflow if inflow > 0 else outflow
    if value > 0:
        labels_with_values.append(f"{node}<br>{value/1e6:.1f} MtC")
    else:
        labels_with_values.append(node)

df_limestone["color"] = df_limestone["Carrier"].map(carrier_colors).fillna("rgba(200,200,200,0.4)")

fig = go.Figure(data=[go.Sankey(
    
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_with_values,
        color="lightgrey"
    ),

    link=dict(
        source=df_limestone["source_id"],
        target=df_limestone["target_id"],
        value=df_limestone["Value_tC"],
        color=df_limestone["color"]
    )
)])

for carrier, color in carrier_colors.items():
    fig.add_trace(go.Scatter(
        x=[None],
        y=[None],
        mode='markers',
        marker=dict(size=10, color=color),
        legendgroup=carrier,
        showlegend=True,
        name=carrier
    ))

fig.update_layout(
    title_text="EU Mineral Carbon Flows (2023)",
    font_size=17,
    width=1700,
    height=1000,
    showlegend=True,
    legend=dict(
        title="Energy carrier",
        orientation="h",
        y=-0.1
    ),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()

## Import of goods

In [18]:
df_goods = pd.read_csv("/Users/charlottewestenberg/Thesis/00data/from_excel/import of goods data barplot.csv")

# Consistent order
# Convert to MtC (optional but recommended)
df_goods["Import"] = df_goods["Import"] / 1e6
df_goods["Export"] = df_goods["Export"] / 1e6

# Optional: sort by imports
df_goods = df_goods.sort_values(by="Import", ascending=False)

# Color palette (reuse or adjust as needed)
colors = {
    "Organic Chemicals": "rgba(60,100,160,0.9)",
    "Pharmaceutical Products": "rgba(210,120,0,0.9)",
    "Chemical products": "rgba(70,250,120,0.9)",
    "Plastic materials & products": "rgba(190, 50, 60, 0.9)",
    "Rubber materials & products": "rgba(80,170,170,0.85)",
    "Textiles": "rgba(150,110,150,0.85)",
    "Toys & sporting goods": "rgba(230,170,180,0.85)"
}


carrier_order = [
    "Organic Chemicals",
    "Chemical products",
    "Plastic materials & products",
    "Rubber materials & products",
    "Pharmaceutical Products",
    "Textiles",
    "Toys & sporting goods"
]

# Create figure
fig = go.Figure()

for _, row in df_goods.iterrows():
    fig.add_bar(
        x=["Import", "Export"],
        y=[row["Import"], row["Export"]],
        name=row["Carrier"],
        marker_color=colors.get(row["Carrier"], "grey")
    )

# Layout
fig.update_layout(
    barmode="stack",
    title="EU Carbon in Imported and Exported Goods (2023)",
    yaxis_title="MtC",
    yaxis=dict(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.25)",  # light grey lines
    gridwidth=1),
    width=800,
    height=800,
    legend_title="Product category",
        legend=dict(
        traceorder="normal",
        orientation="h",     # horizontal
        yanchor="top",
        y=-0.2,              # move below the plot
        xanchor="center",
        x=0.5                # center it
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()

In [16]:
# Create figure
fig = go.Figure()

for _, row in df_goods.iterrows():
    fig.add_bar(
        x=["Import", "Export"],
        y=[row["Import"], row["Export"]],
        name=row["Carrier"],
        marker_color=colors.get(row["Carrier"], "grey")
    )

# Layout
fig.update_layout(
    barmode="group",
    title="EU Carbon in Imported and Exported Goods (2023)",
    yaxis_title="MtC",
    yaxis=dict(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.25)",  # light grey lines
    gridwidth=1),
    width=800,
    height=800,
    legend_title="Product category",
        legend=dict(
        traceorder="normal",
        orientation="h",     # horizontal
        yanchor="top",
        y=-0.2,              # move below the plot
        xanchor="center",
        x=0.5                # center it
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.show()